# **NEURAL NETWORK FOR PHISHING URL ANALYSIS**

---

### **Model Architecture**
The model architecture consists of a combination of a 1D Convolutional Neural Network (CNN) and a Long Short-Term Memory (LSTM) layer.  
The CNN layer acts as a filter, searching for important patterns in the input data. It focuses on small sections at a time and captures specific details within the data.  
The LSTM layer functions like a memory, retaining important recurring patterns in the data sequence.

<style>
.title {
    font-size: 30px;
    color: white;
}
</style>

<div class="title">

---
.

The model's test accuracy is approximately **88.67%** (AUC-ROC 0.946, computed on predicted probabilities over a stratified 20% held-out split).

</div>

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/phishing-detection-rnn-cnn/


## Install required libraries

In [ ]:
!pip install -q tensorflow pandas numpy matplotlib scikit-learn seaborn


# Importing all required libraries
Import necessary libraries such as pandas, numpy, matplotlib, seaborn, and
TensorFlow's Keras module, enabling data manipulation, visualization etc..

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, roc_auc_score


# Data loading and preprocessing

In [ ]:
# Loading the dataset
import os
data_path = 'dataset_phishing.csv' if not IN_COLAB else '/content/drive/MyDrive/phishing-detection-rnn-cnn/dataset_phishing.csv'
data = pd.read_csv(data_path)

data['status'] = data['status'].map({'legitimate': 0, 'phishing': 1})


# Data visualization
## Pie chart to check for any class imbalance

In [ ]:
data['status'].value_counts().plot(kind = 'pie', colors = ['blue', 'green'], labels=['Legitimate', 'Phishing'])

In [ ]:
data['status']

## Train-test data split


In [ ]:
X = data.drop(['url', 'status'], axis = 1)
Y = data['status']
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

# Feature scaling: fit on train only, then transform both splits.
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler so inference reproduces the same preprocessing.
import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

np.save("X_train.npy", X_train)
np.save("Y_train.npy", Y_train)
np.save("X_test.npy", X_test)
np.save("Y_test.npy", Y_test)


## Correlation matrix

In [ ]:
mat = X_train.corr()
plt.figure(figsize=(30, 30))
sns.heatmap(mat, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.xlabel(f'Features (Total: {mat.shape[1]})')
plt.ylabel(f'Features (Total: {mat.shape[1]})')
plt.show()

## Plotting important features


In [ ]:
imp_features = [
    ['length_url', 'nb_dots', 'nb_hyphens', 'nb_at', 'nb_dslash', 'nb_redirection'],
    ['nb_semicolumn', 'nb_www', 'nb_dollar', 'domain_in_title', 'https_token', 'ratio_digits_url', 'iframe'],
    ['nb_comma', 'http_in_path', 'domain_with_copyright', 'domain_age', 'domain_registration_length', 'phish_hints', 'brand_in_path'],
    ['shortest_words_raw', 'shortest_word_host', 'shortest_word_path', 'longest_words_raw', 'longest_word_host', 'longest_word_path'],
    ['avg_words_raw', 'avg_word_host', 'avg_word_path', 'domain_in_brand', 'ratio_intHyperlinks', 'ratio_extHyperlinks', 'ratio_nullHyperlinks']
]

## Model architecture
The model architecture combines a one-dimensional Convolutional Neural Network (1D CNN) with a Long Short-Term Memory (LSTM) layer.  
The primary reason for including the LSTM layer is to identify relevant recurring patterns in the data.  
The model is built using the Keras library with a sequential structure.



In [ ]:
# Build the model — CNN+LSTM hybrid architecture.
# Input shape corrected to (55, 1) to match the dataset's feature count.
# Hidden Dense layers use relu (sigmoid saturated gradients in the original).
# Output layer uses softmax so the two outputs form a proper categorical
# distribution (sum to 1) compatible with sparse_categorical_crossentropy.
model = keras.Sequential([    keras.layers.Conv1D(filters=64, input_shape=(55,1), kernel_size=2, activation='relu'),
    keras.layers.MaxPooling1D(pool_size=2),

    keras.layers.LSTM(100),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation = 'relu'),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dense(64, activation = 'relu'),
    keras.layers.Dense(2, activation='softmax'),
])


## Loss function and model metrics definition
The loss function used here is sparse categorical cross-entropy, which returns one-hot encodings.

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


## Model training
The model is trained with early stopping and learning-rate reduction on plateau.
A 15% validation split is held out from the training set to monitor generalization.


In [ ]:
# CNN+LSTM expects a channel dimension: (N, features, 1)
X_train_3d = X_train_scaled[..., np.newaxis]

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1
    ),
]

history = model.fit(
    X_train_3d, Y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=64,
    callbacks=callbacks,
    verbose=2,
)


## Plotting accuracy vs. epochs graph

In [ ]:
# Persist training history as a JSON dict (the previous pickle format
# stored a Keras History object which broke across TF versions).
import json
with open('history.json', 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in history.history.items()}, f, indent=2)

accuracy = history.history['accuracy']
val_accuracy = history.history.get('val_accuracy', [])

plt.figure(figsize=(8, 6))
plt.plot(accuracy, label='Training Accuracy')
if val_accuracy:
    plt.plot(val_accuracy, label='Validation Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


## Model summary


In [ ]:
model.summary()

## Saving the model
Saving the model helps avoid retraining and allows it to be reused multiple times.

In [ ]:
model.save('my_model.keras')


## Misclassification count
The number of incorrect classifications made by the model on the training data.

In [ ]:
# Misclassification count on the test set.
X_test_3d = X_test_scaled[..., np.newaxis]
Y_pred = model.predict(X_test_3d, verbose=0)
Y_pred_classes = np.argmax(Y_pred, axis=1)
cnt = int(np.sum(Y_pred_classes != np.array(Y_test)))
print(f"Misclassifications on test set: {cnt} / {len(Y_test)}")


## Testing the model
model.evaluate() is a built-in function that helps evaluate the model's test accuracy.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_3d, Y_test)
print(f"Test accuracy: {test_accuracy:.4f}")


## Generating classification reports


In [ ]:
report = classification_report(Y_test, Y_pred_classes)
print(report)


## Confusion matrix

In [ ]:
cm = confusion_matrix(Y_test, Y_pred_classes)
fig, ax = plt.subplots()

sns.heatmap(cm, fmt = 'd', annot=True, cmap='Blues', ax=ax)

ax.set_xlabel('Predicted Class')
ax.set_ylabel('Actual Class')
ax.set_title('Confusion Matrix')


plt.show()

In [ ]:
# ROC curve and AUC computed on predicted probabilities (not argmax).
# The previous version computed AUC on hard predictions, which collapsed
# the confidence signal and produced a degenerate 3-point ROC curve.


## Receiver Operating Characteristic (ROC) curve

In [ ]:
p_phishing = Y_pred[:, 1]
fpr, tpr, thresholds = roc_curve(Y_test, p_phishing)
auc = roc_auc_score(Y_test, p_phishing)

plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], 'k:')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True)
plt.show()


# Custom URL evaluation

In [ ]:
import numpy as np
import re
from urllib.parse import urlparse
import itertools
import pickle

# Load the trained model and the fitted scaler
import tensorflow as tf
model = tf.keras.models.load_model("my_model.keras")
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

def extract_features(url):
    features = {}

    features['length_url'] = len(url)
    parsed = urlparse(url)
    hostname = parsed.netloc
    path = parsed.path
    features['length_hostname'] = len(hostname)

    ip_pattern = r'^(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$'
    features['ip'] = 1 if re.match(ip_pattern, hostname.split(':')[0]) else 0

    features['nb_dots'] = url.count('.')
    features['nb_hyphens'] = url.count('-')
    features['nb_at'] = url.count('@')
    features['nb_qm'] = url.count('?')
    features['nb_and'] = url.count('&')
    features['nb_or'] = url.count('|')
    features['nb_eq'] = url.count('=')
    features['nb_underscore'] = url.count('_')
    features['nb_tilde'] = url.count('~')
    features['nb_percent'] = url.count('%')
    features['nb_slash'] = url.count('/')
    features['nb_star'] = url.count('*')
    features['nb_colon'] = url.count(':')
    features['nb_comma'] = url.count(',')
    features['nb_semicolon'] = url.count(';')
    features['nb_dollar'] = url.count('$')
    features['nb_space'] = url.count(' ')
    features['nb_www'] = 1 if 'www' in hostname.lower() else 0
    features['nb_com'] = 1 if 'com' in hostname.lower() else 0
    features['nb_dslash'] = url.count('//')
    features['http_in_path'] = 1 if 'http' in path.lower() else 0
    features['https_token'] = 1 if url.startswith('https://') else 0

    digits_count = sum(c.isdigit() for c in url)
    features['ratio_digits_url'] = digits_count / len(url) if len(url) > 0 else 0
    digits_count_host = sum(c.isdigit() for c in hostname)
    features['ratio_digits_host'] = digits_count_host / len(hostname) if len(hostname) > 0 else 0

    features['punycode'] = 1 if 'xn--' in hostname.lower() else 0
    features['port'] = 1 if ':' in hostname and any(c.isdigit() for c in hostname.split(':')[1]) else 0

    tlds = ['.com', '.org', '.net', '.edu', '.gov', '.mil', '.int', '.biz', '.info', '.mobi', '.name', '.ly']
    features['tld_in_path'] = 1 if any(tld in path.lower() for tld in tlds) else 0
    features['tld_in_subdomain'] = 1 if hostname.count('.') > 1 and any(tld in hostname.lower().split('.')[0] for tld in tlds) else 0
    features['abnormal_subdomain'] = 1 if hostname.count('.') > 2 else 0
    features['nb_subdomains'] = hostname.count('.')
    features['prefix_suffix'] = 1 if '-' in hostname else 0
    features['random_domain'] = 0

    shortening_services = ['bit.ly', 'goo.gl', 't.co', 'tinyurl.com', 'is.gd',
                          'cli.gs', 'on.ly', 'short.cm', 'tiny.cc', 'shorte.st',
                          'x.co', 'prettylinkpro.com', 'viralurl.com',
                          'qr.net', 'lurl.no', 'tweez.me', 'v.gd', 'tr.im', 'link.zip.net']
    features['shortening_service'] = 1 if any(service in hostname.lower() for service in shortening_services) else 0

    path_extensions = ['.php', '.html', '.htm', '.asp', '.aspx', '.jsp', '.js', '.css', '.py']
    features['path_extension'] = 1 if any(ext in path.lower() for ext in path_extensions) else 0
    features['nb_redirection'] = url.count('http') - 1 if url.count('http') > 1 else 0
    features['nb_external_redirection'] = 0

    raw_words = re.findall(r'[a-zA-Z0-9]+', url)
    host_words = re.findall(r'[a-zA-Z0-9]+', hostname)
    path_words = re.findall(r'[a-zA-Z0-9]+', path) if path else []

    features['length_words_raw'] = len(raw_words)
    features['char_repeat'] = max([len(list(group)) for char, group in itertools.groupby(url)], default=0)
    features['shortest_word_raw'] = min([len(word) for word in raw_words], default=0) if raw_words else 0
    features['shortest_word_host'] = min([len(word) for word in host_words], default=0) if host_words else 0
    features['shortest_word_path'] = min([len(word) for word in path_words], default=0) if path_words else 0
    features['longest_word_raw'] = max([len(word) for word in raw_words], default=0) if raw_words else 0
    features['longest_word_host'] = max([len(word) for word in host_words], default=0) if host_words else 0
    features['longest_word_path'] = max([len(word) for word in path_words], default=0) if path_words else 0
    features['avg_word_raw'] = sum([len(word) for word in raw_words]) / len(raw_words) if raw_words else 0
    features['avg_word_host'] = sum([len(word) for word in host_words]) / len(host_words) if host_words else 0
    features['avg_word_path'] = sum([len(word) for word in path_words]) / len(path_words) if path_words else 0

    phishing_words = ['secure', 'account', 'verify', 'login', 'update', 'signin', 'banking', 'confirm']
    features['phish_hints'] = 1 if any(word in url.lower() for word in phishing_words) else 0
    features['domain_in_brand'] = 0
    features['brand_in_subdomain'] = 0
    features['brand_in_path'] = 0
    features['suspecious_tld'] = 1 if any(hostname.lower().endswith(tld) for tld in ['.tk', '.xyz', '.top', '.ml', '.ga', '.cf', '.gq']) else 0

    ordered_features = [
        features['length_url'], features['length_hostname'], features['ip'],
        features['nb_dots'], features['nb_hyphens'], features['nb_at'],
        features['nb_qm'], features['nb_and'], features['nb_or'],
        features['nb_eq'], features['nb_underscore'], features['nb_tilde'],
        features['nb_percent'], features['nb_slash'], features['nb_star'],
        features['nb_colon'], features['nb_comma'], features['nb_semicolon'],
        features['nb_dollar'], features['nb_space'], features['nb_www'],
        features['nb_com'], features['nb_dslash'], features['http_in_path'],
        features['https_token'], features['ratio_digits_url'],
        features['ratio_digits_host'], features['punycode'], features['port'],
        features['tld_in_path'], features['tld_in_subdomain'],
        features['abnormal_subdomain'], features['nb_subdomains'],
        features['prefix_suffix'], features['random_domain'],
        features['shortening_service'], features['path_extension'],
        features['nb_redirection'], features['nb_external_redirection'],
        features['length_words_raw'], features['char_repeat'],
        features['shortest_word_raw'], features['shortest_word_host'],
        features['shortest_word_path'], features['longest_word_raw'],
        features['longest_word_host'], features['longest_word_path'],
        features['avg_word_raw'], features['avg_word_host'],
        features['avg_word_path'], features['phish_hints'],
        features['domain_in_brand'], features['brand_in_subdomain'],
        features['brand_in_path'], features['suspecious_tld']
    ]
    return np.array(ordered_features, dtype=np.float64)

def predict_url(url):
    features = extract_features(url).reshape(1, -1)
    # Apply the same scaling used during training, then add the channel dim
    features_scaled = scaler.transform(features)[..., np.newaxis]
    prediction = model.predict(features_scaled, verbose=0)
    predicted_class = int(np.argmax(prediction, axis=1)[0])
    probability = float(prediction[0][predicted_class])
    return {
        'url': url,
        'is_phishing': bool(predicted_class),
        'prediction': 'phishing' if predicted_class else 'legitimate',
        'probability': probability
    }

# Function for interactive analysis
def analyze_user_url():
    url = input("\nEnter the URL you want to analyze: ")
    result = predict_url(url)
    print(f"\nURL: {result['url']}")
    print(f"Prediction: {result['prediction']}")
    print(f"Probability: {result['probability']:.4f}")
    if result['is_phishing']:
        print("\nWARNING: This URL has been classified as potentially malicious (phishing).")
        print("It is recommended not to access this website.")
    else:
        print("\nThis URL has been classified as legitimate.")

# Run interactive analysis
analyze_user_url()
